# SitEnt-Sätze in Tabelle exportieren
Pro annotiertem Clause eine Zeile mit:
- Clause
- Ganzer Satz
- SE Type
- Main Referent?
- Verb?

In [1]:
import cassis
import pandas as pd
import os
from tqdm import tqdm

from collections import Counter

In [2]:
folder = "../annotated_corpus/annotated_xmi/"
cas_dict = dict()

with open("".join([folder, "TypeSystem.xml"]), "rb") as f:
    ts = cassis.load_typesystem(f)

for file in tqdm(os.listdir(folder)):
    if file == "TypeSystem.xml":
        continue
    with open("".join([folder, file]), "rb") as f:
        cas = cassis.load_cas_from_xmi(f, typesystem=ts)
    cas_dict[file] = cas
print(len(cas_dict))

100%|██████████| 368/368 [01:12<00:00,  5.07it/s]

367


In [3]:
se_type = ts.get_type('webanno.custom.SituationEntities')
se_feature_type = ts.get_type('webanno.custom.SE_Features')
se_link_type = ts.get_type('webanno.custom.SE_Link')
morph_type = ts.get_type('de.tudarmstadt.ukp.dkpro.core.api.lexmorph.type.morph.MorphologicalFeatures')
sentence_type = ts.get_type('de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Sentence')

In [4]:
def get_fs_by_xmi_id(cas, xmi_id):
    for fs in cas.select_all():  # Durch alle FeatureStructures iterieren
        if getattr(fs, "xmiID", None) == xmi_id:
            return fs
    return None  

verb_features = list()
ref_features = list()
def get_annotated_clauses(cas):
    """iterate over Situtation Entities within sentences

    Args:
        cas (_type_): CAS object

    Returns:
        pd.DataFrame: dataframe with SE per row
    """
    se_list = list()
    for sentence in cas.select(sentence_type):
        for se in cas.select_covered(se_type, sentence):

            se_series = pd.Series({'sentence': sentence.get_covered_text(),
                                   'instanceid': se.instanceid,
                                   'se_xmiID': se.xmiID,
                                   'clause': se.get_covered_text(),
                                   'se_type': se.SE_Type})
            
            # main Referent und Verb erkennen
            se_links = cas.select_covered(se_link_type, se)
            if len(se_links) == 0:
                # TODO: es gibt auch Fälle, in denen kein Link gesetzt ist, weilz.B. nur ein Verb vorkommt
                pass
            elif len(se_links) != 1: # eigentlich sollte es pro SituationEntity nur ein paar aus MainReferent und MainVerb geben
                print(se_links)
            else:
                se_link = se_links[0]
                main_verb = se_link.Governor
                se_series['MainVerb'] = main_verb.get_covered_text()
                for feature in main_verb.Labels.split('|'):
                    feature_split = feature.split('=')
                    se_series[feature_split[0]] = feature_split[1]
                
                main_referent = se_link.Dependent
                se_series['MainReferent'] = main_referent.get_covered_text()
                for feature in main_referent.Labels.split('|'):
                    feature_split = feature.split('=')
                    se_series[feature_split[0]] = feature_split[1]

            se_list.append(se_series.to_frame().T)
    return pd.concat(se_list, ignore_index=True)

anno_list = list()
for cas_name, cas in tqdm(cas_dict.items()):
    df = get_annotated_clauses(cas)
    df['document'] = cas_name
    anno_list.append(df)

clause_df = pd.concat(anno_list, ignore_index=True)
print(len(clause_df))
clause_df.head(20)

100%|██████████| 367/367 [01:30<00:00,  4.04it/s]

49258


,sentence,instanceid,se_xmiID,clause,se_type,MainVerb,Habituality,AspectualClass,MainReferent,Genericity,document
0,I can't believe I wrote all that last year.,blog_Acephalous-Cant-believe.txt_1,231528,I can't believe,STATE,believe,STATIC,STATIVE,I,NON-GENERIC,blog_Acephalous-Cant-believe.txt.xmi
1,I can't believe I wrote all that last year.,blog_Acephalous-Cant-believe.txt_2,231532,I wrote all that last year.,EVENT,wrote,EPISODIC,DYNAMIC,I,NON-GENERIC,blog_Acephalous-Cant-believe.txt.xmi
2,"Acephalous \n\nFriday, 07 May 2010 \n\nBecause...",blog_Acephalous-Cant-believe.txt_3,231536,Acephalous,None,NaN,NaN,NaN,NaN,NaN,blog_Acephalous-Cant-believe.txt.xmi
3,"Acephalous \n\nFriday, 07 May 2010 \n\nBecause...",blog_Acephalous-Cant-believe.txt_4,231537,"Friday, 07 May 2010",None,NaN,NaN,NaN,NaN,NaN,blog_Acephalous-Cant-believe.txt.xmi
4,"Acephalous \n\nFriday, 07 May 2010 \n\nBecause...",blog_Acephalous-Cant-believe.txt_5,231538,Because I've been grading all damn day,STATE,grading,EPISODIC,DYNAMIC,I,NON-GENERIC,blog_Acephalous-Cant-believe.txt.xmi
5,"Acephalous \n\nFriday, 07 May 2010 \n\nBecause...",blog_Acephalous-Cant-believe.txt_6,231542,and am as tired as a Swearengen of hearing oth...,STATE,am,STATIC,STATIVE,I,NON-GENERIC,blog_Acephalous-Cant-believe.txt.xmi
6,"Acephalous \n\nFriday, 07 May 2010 \n\nBecause...",blog_Acephalous-Cant-believe.txt_7,231546,I thought,STATE,thought,STATIC,STATIVE,I,NON-GENERIC,blog_Acephalous-Cant-believe.txt.xmi
7,"Acephalous \n\nFriday, 07 May 2010 \n\nBecause...",blog_Acephalous-Cant-believe.txt_8,231550,that it might be best to avoid jealously,STATE,be,STATIC,STATIVE,it,EXPLETIVE,blog_Acephalous-Cant-believe.txt.xmi
8,"Acephalous \n\nFriday, 07 May 2010 \n\nBecause...",blog_Acephalous-Cant-believe.txt_9,231554,lashing out,None,NaN,NaN,NaN,NaN,NaN,blog_Acephalous-Cant-believe.txt.xmi
9,"Acephalous \n\nFriday, 07 May 2010 \n\nBecause...",blog_Acephalous-Cant-believe.txt_10,231556,"and scribble a ""Best of Acephalous 2009"" post.",None,scribble,NaN,CANNOT_DECIDE,it,CANNOT_DECIDE,blog_Acephalous-Cant-believe.txt.xmi


In [6]:
clause_df.to_excel('../annotated_corpus/se_table.xlsx', sheet_name='situation_entities')

In [8]:
clause_df['se_type'].value_counts(dropna=False)

se_type
STATE                    18113
EVENT                     9636
None                      7589
GENERIC_SENTENCE          7484
REPORT                    1605
GENERALIZING_SENTENCE     1444
QUESTION                   976
IMPERATIVE                 975
CANNOT_DECIDE              959
GENERAL_STATIVE            429
SPEECH_ACT                  48
Name: count, dtype: int64